# CHEOPS PIPE Visualization Tool

## To use it as GUI
- Go to **View**
- Click on **Collapse All Code**

---

**Author:** Sarvesh S. Bhogaokar, Master Student (Looking for a PhD !)  
**Affiliation:** University of Potsdam, Germany  

**Email:** sarvesh1722204@gmail.com, bhogaokar@uni-potsdam.de  

**Co-authors:**  
- Dr. Bruno Merin, ESA  

**Acknowledgements:**  
- Dr. Alexis Brandeker (University of Stockholm, Sweden) 
- Dr. Maximilian Guenther (ESA) 

**Date:** 02 March 2026  
**Version:** 1.0  

---

This notebook is intended for ESA Datalabs users to visualize the phase curves of visits within a PROJECTTYPE-ID using a preprocessing pipeline.

# 🔭 CHEOPS PIPE Visualization Tool – User Guide

---

## 🚀 What This Tool Does

### 📂 1. Data Query & Comparison
- Query CHEOPS visits by **Target Name** or **Program ID** -- ** At the moment only by Target Name **.
- Automatically retrieve both **DRP** and **PIPE** data products.
- Display a side-by-side comparison and identify common or missing visits.


### ⚙️ 2. Preprocessing & Ephemeris Integration
- Apply configurable preprocessing:
  - NMAD-based outlier clipping  
  - Detrending  
  - Residual clipping  
- Query planetary ephemerides from:
  - ExoFOP (TIC / TOI targets)
  - NASA Exoplanet Archive
- Manually override and lock ephemeris parameters per program.

### 📈 3. Visualization
- Generate stacked **phase-folded light curves** per program.
- Plot individual visit light curves.
- Automatically overlay predicted mid-transit markers.
- Organize plots in program-based tabs for structured inspection.

---

## 🧭 How to Use the Tool

### Step 1 — Select Data Source
- Choose your **PIPE data folder**.
- Choose your **CHEOPS DRP folder**.
- Or click **“Use Default Paths”**.

### Step 2 — Run a Query
- Select query mode (Target Name / Program ID).
- Enter the required identifier.
- Click **Run Query** and confirm.

### Step 3 — Select Visits
- Review DRP/PIPE comparison table.
- Select one or multiple visits.
- Click **Confirm Selection**.

### Step 4 — Configure Processing
- Choose DRP aperture and PIPE mode.
- Adjust preprocessing parameters if needed.
- Click **Preprocess** to lock settings.

### Step 5 — Add Ephemerides
- Click **Query Ephemerides** to auto-fill period & mid-transit time.
- Or enable manual mode and enter values.
- Click **Save Ephemeris**.

### Step 6 — Plot Lightcurves
- Click **Plot DRP LC** or **Plot PIPE LC**.
- Explore:
  - Stacked phase-folded plot
  - Individual visit light curves
  - Program-specific tabs

---

ℹ️ **Tip:** Collapse all code cells (View → Collapse All Code) to use this notebook as a GUI interface.


In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
#from ipysplitpanes import SplitPanes
from ipyfilechooser import FileChooser
#from ipydatagrid import DataGrid
from pathlib import Path
import pandas as pd
import sys, os
import warnings
import re
from ipywidgets import GridspecLayout
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
## ===============================
# 1. IMPORTS & CONFIGURATION
# ===============================

import ipywidgets as widgets
from IPython.display import display, clear_output
from ipysplitpanes import SplitPanes
from ipyfilechooser import FileChooser
from ipydatagrid import DataGrid
from pathlib import Path
import pandas as pd
import sys, os
import warnings
import re
from ipywidgets import GridspecLayout
import numpy as np
import matplotlib.pyplot as plt
import time

if "/media/backend" not in sys.path:
    sys.path.append("/media/backend")

if "/media/Ephemerides" not in sys.path:
    sys.path.append("/media/Ephemerides")



pd.set_option("display.float_format", "{:.6f}".format)

warnings.filterwarnings("ignore")


%load_ext autoreload
%autoreload 1

# sys.path.append(os.path.abspath('backend'))
%aimport query_table
%aimport preprocessing
%aimport phase_folding

plt.rcParams.update({
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})


DEFAULT_DRP_PATH = "/data/user/che_data_dev/repo"
DEFAULT_PIPE_PATH = "/data/user/che_pipe/Output_lc/visits"


# ===============================
# 2. APPLICATION STATE
# ===============================

STATE = {}
INPUT_WIDGETS = {}


# ===============================
# 3. MAIN APP CONTAINER
# ===============================

app_container = widgets.VBox()
display(app_container)


# ===============================
# 4. VIEW LAYER (UI BUILDERS)
# ===============================
def build_base_layout():
    return [
        data_section,
        query_section,
        results_area
    ]

def build_data_source_section():

    heading = widgets.HTML("<h3>Data Source Settings</h3>")

    pipe_chooser = FileChooser(path=".", title="Select PIPE Data Folder", show_only_dirs=True)
    cheops_chooser = FileChooser(path=".", title="Select CHEOPS DRP Folder", show_only_dirs=True)

    default_button = widgets.Button(description="Use Default Paths", button_style="info")
    default_output = widgets.Output()

    container = widgets.VBox([
        heading,
        pipe_chooser,
        cheops_chooser,
        default_button,
        default_output
    ])

    return container, pipe_chooser, cheops_chooser, default_button, default_output


def build_query_section():

    heading = widgets.HTML("<h3>Query Section</h3>")

    query_mode = widgets.Dropdown(
        options=[
            ("Select From", "no_query"),
            ("Target Name", "target"),
            ("Program ID", "program"),
            ("Program ID + Visit Number", "program_visit"),
            ("Obs Request ID", "obs_request")
        ],
        description="Query by:",
        layout=widgets.Layout(width="40%")
    )

    dynamic_area = widgets.Output()

    run_button = widgets.Button(
        description="Run the Query",
        button_style="success",
        disabled=True
    )

    container = widgets.VBox([
        heading,
        query_mode,
        dynamic_area,
        run_button
    ])

    return container, query_mode, dynamic_area, run_button

def build_confirmation_section(temp_state, on_confirm_callback):

    description = widgets.HTML(
        value=f"""
        <b>Confirmation Required</b><br><br>
        You are about to run the query with:<br>
        <pre>{temp_state}</pre>
        """
    )

    confirm_button = widgets.Button(
        description="Yes, Run Query",
        button_style="success"
    )

    cancel_button = widgets.Button(
        description="Cancel",
        button_style="danger"
    )

    button_box = widgets.HBox([confirm_button, cancel_button])

    container = widgets.VBox([description, button_box])

    # Attach callbacks
    confirm_button.on_click(lambda b: on_confirm_callback())
    cancel_button.on_click(lambda b: restore_main_layout())

    return container


def build_results_section(df_drp, df_pipe, df_combined):

    title_drp = widgets.HTML("<h3>DRP Results</h3>")
    title_pipe = widgets.HTML("<h3>PIPE Results</h3>")
    title_combined = widgets.HTML("<h3>Combined Comparison</h3>")

    out_drp = widgets.Output()
    out_pipe = widgets.Output()
    out_combined = widgets.Output()

    with out_drp:
        display(df_drp)

    with out_pipe:
        display(df_pipe)

    with out_combined:
        display(df_combined)

    # side_by_side = widgets.HBox([
    #     widgets.VBox([title_drp, out_drp]),
    #     widgets.VBox([title_pipe, out_pipe])
    # ])
    
    left_box = widgets.VBox(
        [title_drp, out_drp],
        layout=widgets.Layout(
            flex='1 1 50%',
            border='1px solid #ccc',
            overflow='auto'
        )
    )

    right_box = widgets.VBox(
        [title_pipe, out_pipe],
        layout=widgets.Layout(
            flex='1 1 50%',
            border='1px solid #ccc',
            overflow='auto'
        )
    )

    side_by_side = widgets.HBox(
        [left_box, right_box],
        layout=widgets.Layout(width='100%')
    )

    container = widgets.VBox([
        side_by_side,
        title_combined,
        out_combined
    ])

    return container


def build_visit_selection(df_combined):

    selection_title = widgets.HTML("<h3>Select Visits to Plot</h3>")

    visit_labels = {}
    for _, row in df_combined.iterrows():
        visit_id = row["visit_id"]
        drp = row.get("drp_exists", False)
        pipe = row.get("pipe_exists", False)

        if drp and pipe:
            status = "✔ COMMON"
        elif drp:
            status = "⚠ DRP"
        elif pipe:
            status = "◯ PIPE"
        else:
            status = ""

        label = (
            f"{status} | "
            f"PT={row['program_type']} | "
            f"PID={row['program_id']} | "
            f"REQ={row['req_id']} | "
            f"{row['target']} | "
            f"VisitID={visit_id}"
        )

        visit_labels[label] = visit_id
    visit_selector = widgets.SelectMultiple(
        options=visit_labels,
        description="Visits:",
        layout=widgets.Layout(width="60%", height="250px")
    )
    
    
    toggle_button = widgets.Button(
        description="Select All",
        button_style="info")    
    confirm_button = widgets.Button(description="Confirm Selection", button_style="success")
    selection_output = widgets.Output()

    container = widgets.VBox([
        selection_title,
        visit_selector,
        toggle_button,
        confirm_button,
        selection_output
    ])

    def confirm_selection(b):

        selected_visits = list(visit_selector.value)

        with selection_output:
            clear_output()

            if not selected_visits:
                print("Please select at least one visit.")
                STATE["selected_visits"] = None
                update_plot_settings_enabled()
                return

            for prefix in ["DRP", "PIPE"]:

                if STATE.get(f"{prefix}_preprocessing_done"):

                    # Reset preprocessing state
                    STATE[f"{prefix}_preprocessing_done"] = False

                    # Re-enable controls
                    controls = STATE.get(f"{prefix.lower()}_preproc_controls")
                    if controls:

                        for key in [
                            "no_preproc",
                            "outlier",
                            "detrend",
                            "residual",
                            "preprocess_btn"
                        ]:
                            controls[key].disabled = False

                        controls["reset_preproc_btn"].disabled = True

            STATE["selected_visits"] = selected_visits

            selected_df = df_combined[
                df_combined["visit_id"].isin(selected_visits)
            ].reset_index(drop=True)

            STATE["df_selected"] = selected_df

            display(widgets.HTML("<b>Selected Visits:</b>"))
            display(selected_df)
            
            # Build ephemeris tables after selection
            build_ephemeris_rows("DRP")
            build_ephemeris_rows("PIPE")

            update_plot_settings_enabled()

            
           
    
    all_visit_ids = list(visit_labels.values())

    def toggle_selection(b):

        # If everything already selected → deselect
        if len(visit_selector.value) == len(all_visit_ids):
            visit_selector.value = ()
            toggle_button.description = "Select All"

        # Otherwise → select all
        else:
            visit_selector.value = tuple(all_visit_ids)
            toggle_button.description = "Deselect All"
    

    confirm_button.on_click(confirm_selection)
    toggle_button.on_click(toggle_selection)

    return container


def build_plot_settings():

    # ----- DRP SECTION -----
    drp_title = widgets.HTML("<h4>Select DRP Aperture</h4>")

    aperture_options = []
    for px in range(15, 41):
        if px == 25:
            aperture_options.append(("25 px (Default)", 'DEFAULT'))
        else:
            aperture_options.append((f"{px} px", px))

    drp_dropdown = widgets.Dropdown(
        options=aperture_options,
        value='DEFAULT',
        layout=widgets.Layout(width="200px"),
        disabled= True
    )

    drp_box = widgets.VBox([
        drp_title,
        drp_dropdown
    ], layout=widgets.Layout(
        border="1px solid #ccc",
        padding="10px",
        width="50%"
    ))

    # ----- PIPE SECTION -----
    pipe_title = widgets.HTML("<h4>Select PIPE Data Type</h4>")

    pipe_dropdown = widgets.Dropdown(
        options=[
            ("SA only", "sa"),
            ("IM only", "im"),
            ("SA or IM", "either")
        ],
        value="either",
        layout=widgets.Layout(width="200px"),
        disabled=True
    )

    pipe_box = widgets.VBox([
        pipe_title,
        pipe_dropdown
    ], layout=widgets.Layout(
        border="1px solid #ccc",
        padding="10px",
        width="50%",
    ))


    settings_hbox = widgets.HBox([
        drp_box,
        pipe_box
    ], layout=widgets.Layout(width="100%"))

    return settings_hbox, drp_dropdown, pipe_dropdown


def build_preprocessing_section():
    
    title = widgets.HTML("<h2>Preprocessing Pipeline</h2>")
    
    # -----------------------------
    # Helper to build one side (DRP or PIPE)
    # -----------------------------
    
    def build_single_preprocessing_box(label_prefix):

        section_title = widgets.HTML(f"<h4>{label_prefix} Preprocessing</h4>")

        # ----------------------
        # Default configuration
        # ----------------------

        DEFAULTS = {
            "no_preproc": False,
            "outlier": True,
            "nmad_outlier": 15,
            "detrend": True,
            "tdens": 20,
            "residual": True,
            "nmad_residual": 15
        }

        # ----------------------
        # Widgets
        # ----------------------

        no_preproc_cb = widgets.Checkbox(
            value=DEFAULTS["no_preproc"],
            description="No preprocessing",
            disabled=True
        )

        outlier_cb = widgets.Checkbox(
            value=DEFAULTS["outlier"],
            description="Apply outlier clipping",
            disabled=True
        )

        nmad_outlier = widgets.IntText(
            value=DEFAULTS["nmad_outlier"],
            description="Nmad:",
            layout=widgets.Layout(width="180px"),
            disabled=True
        )

        outlier_default = widgets.HTML("<span style='color:gray'>Default = 15</span>")
        outlier_box = widgets.HBox([nmad_outlier, outlier_default])

        detrend_cb = widgets.Checkbox(
            value=DEFAULTS["detrend"],
            description="Apply detrending",
            disabled=True
        )

        tdens_dropdown = widgets.Dropdown(
            options=list(range(5, 36)),
            value=DEFAULTS["tdens"],
            description="tdens:",
            layout=widgets.Layout(width="200px"),
            disabled=True
        )

        detrend_default = widgets.HTML("<span style='color:gray'>Default = 20</span>")
        detrend_box = widgets.HBox([tdens_dropdown, detrend_default])

        residual_cb = widgets.Checkbox(
            value=DEFAULTS["residual"],
            description="Apply residual clipping",
            disabled=True
        )

        nmad_residual = widgets.IntText(
            value=DEFAULTS["nmad_residual"],
            description="Nmad:",
            layout=widgets.Layout(width="180px"),
            disabled=True
        )

        residual_default = widgets.HTML("<span style='color:gray'>Default = 15</span>")
        residual_box = widgets.HBox([nmad_residual, residual_default])

        reset_button = widgets.Button(
            description="Reset Default Settings",
            button_style="warning",
            disabled=True,
            layout=widgets.Layout(width="250px")
        )

        preprocess_button = widgets.Button(
            description=f"Preprocess {label_prefix} Visits",
            button_style="primary",
            disabled=True,
            layout=widgets.Layout(width="250px")
        )
        
        reset_preprocessing_button = widgets.Button(
        description="Reset Preprocessing",
        button_style="danger",
        disabled=True,
        layout=widgets.Layout(width="250px")
        )
        
        progress_output = widgets.Output(layout=widgets.Layout(height="120px", overflow="auto"))

        # ----------------------
        # Logic Functions
        # ----------------------

        def update_parameter_enable_state(change=None):

            if no_preproc_cb.value:

                # Force uncheck others
                outlier_cb.value = False
                detrend_cb.value = False
                residual_cb.value = False

                # Disable everything else
                outlier_cb.disabled = True
                detrend_cb.disabled = True
                residual_cb.disabled = True

                nmad_outlier.disabled = True
                tdens_dropdown.disabled = True
                nmad_residual.disabled = True

            else:
                outlier_cb.disabled = False
                detrend_cb.disabled = False
                residual_cb.disabled = False

                nmad_outlier.disabled = not outlier_cb.value
                tdens_dropdown.disabled = not detrend_cb.value
                nmad_residual.disabled = not residual_cb.value

            # ALWAYS re-evaluate reset state
            check_if_modified()
            
            
        def check_if_modified(change=None):

            is_default = (
                no_preproc_cb.value == DEFAULTS["no_preproc"] and
                outlier_cb.value == DEFAULTS["outlier"] and
                nmad_outlier.value == DEFAULTS["nmad_outlier"] and
                detrend_cb.value == DEFAULTS["detrend"] and
                tdens_dropdown.value == DEFAULTS["tdens"] and
                residual_cb.value == DEFAULTS["residual"] and
                nmad_residual.value == DEFAULTS["nmad_residual"]
            )

            reset_button.disabled = is_default

        def reset_defaults(b):

            no_preproc_cb.value = DEFAULTS["no_preproc"]
            outlier_cb.value = DEFAULTS["outlier"]
            nmad_outlier.value = DEFAULTS["nmad_outlier"]
            detrend_cb.value = DEFAULTS["detrend"]
            tdens_dropdown.value = DEFAULTS["tdens"]
            residual_cb.value = DEFAULTS["residual"]
            nmad_residual.value = DEFAULTS["nmad_residual"]

            update_parameter_enable_state()
            check_if_modified()
        
        def lock_preprocessing(b):

            df_selected_full = STATE["df_selected"]
            if label_prefix == "DRP":

                df_selected = df_selected_full.copy()

            else:  # PIPE

                if "pipe_exists" in df_selected_full.columns:
                    df_selected = df_selected_full[
                        df_selected_full["pipe_exists"] == True
                    ].copy()
                else:
                    df_selected = df_selected_full.copy()

            controls = STATE[f"{label_prefix.lower()}_preproc_controls"]

            settings_from_ui = {
                "no_preproc": controls["no_preproc"].value,
                "apply_outlier": controls["outlier"].value,
                "nmad_outlier": controls["nmad_outlier"].value,
                "apply_detrend": controls["detrend"].value,
                "tdens": controls["tdens"].value,
                "apply_residual": controls["residual"].value,
                "nmad_residual": controls["nmad_residual"].value
            }

            # ------------------------------------------
            # Run backend preprocessing
            # ------------------------------------------

            if label_prefix == "DRP":

                aperture = STATE["drp_aperture_widget"].value
                
                with progress_output:
                    clear_output()
                
                
                    df = preprocessing.run_preprocessing(
                        df_selected,
                        prefix="DRP",
                        aperture=aperture,
                        settings=settings_from_ui
                    )

                STATE["DRP_processed_df"] = df
                STATE[f"DRP_processed_version"] = time.time()


            else:

                pipe_mode = STATE["pipe_mode_widget"].value
                
                with progress_output:
                    clear_output()

                    df = preprocessing.run_preprocessing(
                        df_selected,
                        prefix="PIPE",
                        pipe_mode=pipe_mode,
                        settings=settings_from_ui
                    )

                STATE["PIPE_processed_df"] = df
            
                STATE[f"PIPE_processed_version"] = time.time()
            # ------------------------------------------
            # Lock UI
            # ------------------------------------------

            for widget in [
                controls["no_preproc"],
                controls["outlier"],
                controls["nmad_outlier"],
                controls["detrend"],
                controls["tdens"],
                controls["residual"],
                controls["nmad_residual"],
                controls["reset"],
                controls["preprocess_btn"]
            ]:
                widget.disabled = True

            controls["reset_preproc_btn"].disabled = False

            STATE[f"{label_prefix}_preprocessing_done"] = True

            update_plot_settings_enabled()
        
        def unlock_preprocessing(b):

            has_selection = STATE.get("selected_visits") is not None

            no_preproc_cb.disabled = not has_selection
            outlier_cb.disabled = not has_selection
            detrend_cb.disabled = not has_selection
            residual_cb.disabled = not has_selection
            preprocess_button.disabled = not has_selection

            update_parameter_enable_state()

            reset_preprocessing_button.disabled = True

            STATE[f"{label_prefix}_preprocessing_done"] = False



            if f"{label_prefix}_ephem_query_button" in STATE:
                STATE[f"{label_prefix}_ephem_query_button"].disabled = True

            update_plot_settings_enabled()
        

        # ----------------------
        # Observers
        # ----------------------

        no_preproc_cb.observe(update_parameter_enable_state, names="value")
        outlier_cb.observe(update_parameter_enable_state, names="value")
        detrend_cb.observe(update_parameter_enable_state, names="value")
        residual_cb.observe(update_parameter_enable_state, names="value")
        nmad_outlier.observe(check_if_modified, names="value")
        tdens_dropdown.observe(check_if_modified, names="value")
        nmad_residual.observe(check_if_modified, names="value")
        no_preproc_cb.observe(check_if_modified, names="value")
        outlier_cb.observe(check_if_modified, names="value")
        detrend_cb.observe(check_if_modified, names="value")
        residual_cb.observe(check_if_modified, names="value")

        reset_button.on_click(reset_defaults)
        preprocess_button.on_click(lock_preprocessing)
        reset_preprocessing_button.on_click(unlock_preprocessing)
        
        update_parameter_enable_state()
        check_if_modified()


        # ----------------------
        # Layout
        # ----------------------
        
        processing_box = widgets.HBox([preprocess_button,reset_preprocessing_button], layout=widgets.Layout(
            border="1px solid #ccc",
            padding="10px",
            width="50%"
        )
                                     
                                     )

        box = widgets.VBox([
            section_title,
            no_preproc_cb,
            outlier_cb,
            outlier_box,
            detrend_cb,
            detrend_box,
            residual_cb,
            residual_box,
            reset_button,
            processing_box,
            progress_output 
        ], layout=widgets.Layout(
            border="1px solid #ccc",
            padding="10px",
            width="50%"
        ))

        controls = {
            "no_preproc": no_preproc_cb,
            "outlier": outlier_cb,
            "nmad_outlier": nmad_outlier,
            "detrend": detrend_cb,
            "tdens": tdens_dropdown,
            "residual": residual_cb,
            "nmad_residual": nmad_residual,
            "reset": reset_button,
            "preprocess_btn": preprocess_button,
            "reset_preproc_btn": reset_preprocessing_button
        }

        return box, controls

    
    # -----------------------------
    # Build both sides
    # -----------------------------
    
    drp_box, drp_controls = build_single_preprocessing_box("DRP")
    pipe_box, pipe_controls = build_single_preprocessing_box("PIPE")

    STATE["drp_preproc_controls"] = drp_controls
    STATE["pipe_preproc_controls"] = pipe_controls
    
    hbox = widgets.HBox([drp_box, pipe_box],
                        layout=widgets.Layout(width="100%"))
    
    container = widgets.VBox([title, hbox])
    
    return container

    
def build_ephemerides_section():

    title = widgets.HTML("<h2>Ephemerides</h2>")

    # Build independent DRP + PIPE boxes
    drp_box = build_single_ephemeris_box("DRP")
    pipe_box = build_single_ephemeris_box("PIPE")

    hbox = widgets.HBox(
        [drp_box, pipe_box],
        layout=widgets.Layout(width="100%")
    )

    return widgets.VBox([title, hbox])


def build_single_ephemeris_box(prefix):

    # ------------------------------
    # UI Elements
    # ------------------------------

    section_title = widgets.HTML(f"<h3>{prefix} Ephemerides</h3>")

    table_box = widgets.VBox()
    table_box.layout = widgets.Layout(
    max_height="350px",
    overflow="auto"
    )
    query_button = widgets.Button(
        description=f"Query {prefix} Ephemerides",
        button_style="info",
        disabled=True,
        layout=widgets.Layout(width="250px")
    )

    output_area = widgets.Output()

    container = widgets.VBox(
        [section_title, table_box, query_button, output_area],
        layout=widgets.Layout(
            border="1px solid #ccc",
            padding="10px",
            width="50%"
        )
    )

    # ------------------------------
    # Store Independent STATE Keys
    # ------------------------------

    STATE[f"{prefix}_ephem_table_box"] = table_box
    STATE[f"{prefix}_ephem_query_button"] = query_button
    STATE[f"{prefix}_ephem_output"] = output_area
    STATE[f"{prefix}_ephem_row_widgets"] = []
    
    query_button.on_click(
    lambda b, p=prefix: run_ephemeris_query(b, p)
    )

    save_button = widgets.Button(
    description="Save Ephemeris",
    button_style="success",
    disabled=True,
    layout=widgets.Layout(width="200px"))

    reset_button = widgets.Button(
        description="Reset Ephemeris",
        button_style="warning",
        disabled=True,
        layout=widgets.Layout(width="200px")
    )

    button_row = widgets.HBox([query_button, save_button, reset_button])

    container = widgets.VBox(
        [section_title, table_box, button_row, output_area], layout=widgets.Layout( border="1px solid #ccc", padding="10px", width="50%" ) )
        
    STATE[f"{prefix}_ephem_save_button"] = save_button
    STATE[f"{prefix}_ephem_reset_button"] = reset_button
        
    def save_ephemeris(prefix):

        row_widgets = STATE.get(f"{prefix}_ephem_row_widgets", [])

        saved_rows = []
        enable_states = []

        for r in row_widgets:

            # 🔹 Store previous enable state
            enable_states.append(r["enable_cb"].value)

            saved_rows.append({
                "program_type": r["program_type"],
                "program_id": r["program_id"],
                "req_id": r["req_id"],
                "period_days": r["period_box"].value,
                "mid_transit_time": r["mid_box"].value
            })

            r["enable_cb"].value = True

            r["manual_target"].disabled = True
            r["period_box"].disabled = True
            r["mid_box"].disabled = True
            r["enable_cb"].disabled = True

        saved_df = pd.DataFrame(saved_rows)

        STATE[f"{prefix}_ephem_saved"] = saved_df
        STATE[f"{prefix}_ephem_previous_enable_state"] = enable_states

        STATE[f"{prefix}_ephem_query_button"].disabled = True
        STATE[f"{prefix}_ephem_save_button"].disabled = True
        STATE[f"{prefix}_ephem_reset_button"].disabled = False
        
        if f"{prefix}_lc_plot_button" in STATE:
            STATE[f"{prefix}_lc_plot_button"].disabled = False

        print("Ephemeris saved and locked.")
        
    save_button.on_click(lambda b, p=prefix: save_ephemeris(p))
        
    def reset_ephemeris(prefix):

        STATE[f"{prefix}_ephem_saved"] = None

        row_widgets = STATE.get(f"{prefix}_ephem_row_widgets", [])
        previous_states = STATE.get(f"{prefix}_ephem_previous_enable_state", [])

        for i, r in enumerate(row_widgets):

            r["enable_cb"].disabled = False

            # Restore previous enable state
            if i < len(previous_states):
                r["enable_cb"].value = previous_states[i]
            else:
                r["enable_cb"].value = False

            # Apply field enable logic based on checkbox state
            if r["enable_cb"].value:
                r["manual_target"].disabled = False
                r["period_box"].disabled = False
                r["mid_box"].disabled = False
            else:
                r["manual_target"].disabled = True
                r["period_box"].disabled = True
                r["mid_box"].disabled = True

        STATE[f"{prefix}_ephem_query_button"].disabled = False
        STATE[f"{prefix}_ephem_save_button"].disabled = False
        STATE[f"{prefix}_ephem_reset_button"].disabled = True
        
        # Disable plot button again
        if f"{prefix}_lc_plot_button" in STATE:
            STATE[f"{prefix}_lc_plot_button"].disabled = True

        print("Ephemeris unlocked and previous states restored.")
    reset_button.on_click(lambda b, p=prefix: reset_ephemeris(p))
        
    return container



def build_ephemeris_rows(prefix):

    df_sel = STATE.get("df_selected", pd.DataFrame())
    table_box = STATE[f"{prefix}_ephem_table_box"]

    if df_sel.empty:
        table_box.children = []
        return

    unique_programs = (
        df_sel[["program_type", "program_id", "req_id","target"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    n_rows = len(unique_programs) + 1  # + header
    n_cols = 8

    grid = GridspecLayout(n_rows, n_cols)

    # Define strict column widths
    column_widths = [
        "110px",  # Program Type
        "90px",   # Program ID
        "90px",   # Req ID
        "150px",  # Target Name
        "150px",  # Manual Target
        "110px",  # Period
        "130px",  # Mid Transit
        "90px"    # Enable
    ]

    grid.layout.grid_template_columns = " ".join(column_widths)
    grid.layout.align_items = "center"
    grid.layout.justify_items = "center"
    # -----------------------
    # Header Row
    # -----------------------

    headers = [
        "Program Type",
        "Program ID",
        "Req ID",
        "Target Name",
        "Manual Target",
        "Period (days)",
        "Mid Transit",
        "Enable Manual"
    ]

    for col, title in enumerate(headers):
        grid[0, col] = widgets.HTML(
            f"<b>{title}</b>",
            layout=widgets.Layout(
                padding="4px",
                width=column_widths[col]
            )
        )
    row_widgets = []

    # -----------------------
    # Data Rows
    # -----------------------

    for i, row in unique_programs.iterrows():

        r = i + 1

        program_type = str(row["program_type"])
        program_id = str(row["program_id"])
        req_id = str(row["req_id"])
        target_name = str(row["target"])

        grid[r, 0] = widgets.Label(program_type)
        grid[r, 1] = widgets.Label(program_id)
        grid[r, 2] = widgets.Label(req_id)
        grid[r, 3] = widgets.Label(target_name)

        manual_target = widgets.Text(
        value=target_name,
        disabled=True,
        layout=widgets.Layout(width="140px")
        )

        period_box = widgets.FloatText(
            disabled=True,
            layout=widgets.Layout(width="100px")
        )
        
        mid_box = widgets.FloatText(
            disabled=True,
            layout=widgets.Layout(width="120px")
        )
        
        enable_cb = widgets.Checkbox(
        value=False,
        indent=False,
        layout=widgets.Layout(width="70px")
        )

        # Manual toggle logic
        def make_toggle(manual_target, period_box, mid_box):
            def toggle(change):
                enabled = change["new"]
                manual_target.disabled = not enabled
                period_box.disabled = not enabled
                mid_box.disabled = not enabled
            return toggle

        enable_cb.observe(
            make_toggle(manual_target, period_box, mid_box),
            names="value"
        )

        grid[r, 4] = manual_target
        grid[r, 5] = period_box
        grid[r, 6] = mid_box
        grid[r, 7] = enable_cb

        row_widgets.append({
            "program_type": program_type,
            "program_id": program_id,
            "req_id": req_id,
            "target_name": target_name,
            "manual_target": manual_target,
            "period_box": period_box,
            "mid_box": mid_box,
            "enable_cb": enable_cb
        })

    STATE[f"{prefix}_ephem_row_widgets"] = row_widgets
    table_box.children = [grid]
    
def build_lightcurve_section():

    title = widgets.HTML("<h2>Lightcurve Plots</h2>")

    drp_box = build_single_lc_box("DRP")
    pipe_box = build_single_lc_box("PIPE")

    container = widgets.VBox([
        title,
        widgets.HBox(
            [drp_box, pipe_box],
            layout=widgets.Layout(width="100%")
        )
    ])

    return container

def build_single_lc_box(prefix):

    plot_button = widgets.Button(
        description=f"Plot {prefix} LC",
        button_style="success",
        disabled=True,  # initially disabled
        layout=widgets.Layout(width="200px")
    )

    main_tab = widgets.Tab()
    main_tab.children = []

    STATE[f"{prefix}_lc_plot_button"] = plot_button
    STATE[f"{prefix}_lc_main_tab"] = main_tab
    
    plot_button.on_click(
    lambda b, p=prefix: build_lc_tabs(p))

    box = widgets.VBox(
        [plot_button, main_tab],
        layout=widgets.Layout(
            border="1px solid #ccc",
            padding="10px",
            width="50%"
        )
    )

    return box



def build_lc_tabs(prefix):

    df_sel = STATE.get(f"{prefix}_processed_df")
    ephem_df = STATE.get(f"{prefix}_ephem_saved")
    main_tab = STATE.get(f"{prefix}_lc_main_tab")

    if main_tab is None:
        return

    # -------------------------------------------------
    # No processed data
    # -------------------------------------------------
    if df_sel is None or df_sel.empty:
        main_tab.children = [
            widgets.HTML("<h3 style='color:red;'>No processed data available.</h3>")
        ]
        main_tab.set_title(0, "Empty")
        return

    # -------------------------------------------------
    # No ephemeris
    # -------------------------------------------------
    if ephem_df is None or ephem_df.empty:
        main_tab.children = [
            widgets.HTML("<h3 style='color:red;'>No saved ephemeris.</h3>")
        ]
        main_tab.set_title(0, "No Ephemeris")
        return

    # -------------------------------------------------
    # Loading screen
    # -------------------------------------------------
    loading_box = widgets.VBox([
        widgets.HTML("<h3 style='color:#2c7be5;'>Computing phase folds... please wait</h3>")
    ])
    main_tab.children = [loading_box]
    main_tab.set_title(0, "Loading")

    # -------------------------------------------------
    # Smart cache check
    # -------------------------------------------------
    current_version = STATE.get(f"{prefix}_processed_version")
    cached_version = STATE.get(f"{prefix}_phase_cache_version")
    phase_cache = STATE.get(f"{prefix}_phase_cache")

    if phase_cache is None or cached_version != current_version:

        phase_cache = phase_folding.precompute_phase_all_programs(
            df_sel,
            ephem_df,
            prefix
        )

        if not phase_cache:
            main_tab.children = [
                widgets.HTML("<h3 style='color:red;'>Phase computation failed.</h3>")
            ]
            main_tab.set_title(0, "Error")
            return

        STATE[f"{prefix}_phase_cache"] = phase_cache
        STATE[f"{prefix}_phase_cache_version"] = current_version

    # -------------------------------------------------
    # Group once (important optimization)
    # -------------------------------------------------
    grouped = dict(tuple(df_sel.groupby(["program_type", "program_id","req_id"])))

    unique_programs = list(grouped.keys())

    if not unique_programs:
        main_tab.children = [
            widgets.HTML("<h3>No programs found.</h3>")
        ]
        main_tab.set_title(0, "Empty")
        return

    tab_children = []
    tab_titles = []

    # ==================================================
    # ALL PROGRAMS TAB
    # ==================================================
    all_programs_container = widgets.VBox()
    program_sections = []

    for (program_type, program_id,req_id), df_prog in grouped.items():

        df_prog = df_prog.sort_values("visit_id")

        program_box = build_program_plot_box(
            prefix,
            program_type,
            program_id,
            req_id,
            df_prog
        )

        program_sections.append(program_box)

    all_programs_container.children = program_sections
    tab_children.append(all_programs_container)
    tab_titles.append("All Programs")

    # ==================================================
    # INDIVIDUAL PROGRAM TABS
    # ==================================================
    for (program_type, program_id,req_id), df_prog in grouped.items():

        df_prog = df_prog.sort_values("visit_id")

        program_box = build_program_plot_box(
            prefix,
            program_type,
            program_id,
            req_id,
            df_prog
        )

        tab_children.append(program_box)
        tab_titles.append(f"{program_type}-{program_id}-{req_id}")

    # -------------------------------------------------
    # Finalize tabs
    # -------------------------------------------------
    main_tab.children = tab_children

    for i, title in enumerate(tab_titles):
        main_tab.set_title(i, title)
        
        

def build_program_plot_box(prefix, program_type, program_id, req_id,df_prog):

    title = widgets.HTML(f"<h4>{program_type} - {program_id} - {req_id}</h4>")

    if df_prog is None or df_prog.empty:
        return widgets.VBox([
            title,
            widgets.HTML("<b>No visit data available.</b>")
        ])

    # -------------------------------------------------
    # Get cached phase data
    # -------------------------------------------------
    phase_cache = STATE.get(f"{prefix}_phase_cache", {})
    phase_data = phase_cache.get((program_type, program_id,req_id))

    # -------------------------------------------------
    # Get ephemeris row once (important optimization)
    # -------------------------------------------------
    ephem_df = STATE.get(f"{prefix}_ephem_saved")
    ephem_row = None

    if ephem_df is not None and not ephem_df.empty:
        ephem_row = ephem_df[
            (ephem_df["program_type"].astype(str).str.strip() == str(program_type).strip()) &
            (ephem_df["program_id"].astype(str).str.strip() == str(program_id).strip()) &
            (ephem_df["req_id"].astype(str).str.strip() == str(req_id).strip())
        ]

    # =====================================================
    # STACK PHASE FOLD
    # =====================================================
    phase_output = widgets.Output(
        layout=widgets.Layout(
            height="450px",
            border="1px solid #bbb",
            margin="15px 0px 30px 0px",
            padding="15px"
        )
    )

    with phase_output:

        if phase_data is None:
            display(widgets.HTML("<b>No phase data available.</b>"))
        else:
            phase, flux = phase_data

            plt.figure(figsize=(6, 4))
            plt.scatter(phase, flux, s=6, alpha=0.6)
            plt.axvline(0, color="crimson", linewidth=2)
            plt.xlabel("Phase")
            plt.ylabel("Normalized Flux")
            plt.title("Stacked Phase Folded Lightcurve")
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()

    # =====================================================
    # VISIT LIGHTCURVES
    # =====================================================
    visit_outputs = []

    for _, row in df_prog.iterrows():

        visit_out = widgets.Output(
            layout=widgets.Layout(
                width="550px",
                border="1px solid #e0e0e0",
                padding="10px",
                margin="0px 0px 15px 0px"
            )
        )

        with visit_out:

            time_arr = row["time"]
            flux_arr = row["flux"]

            t0_plot = time_arr.min()
            time_rel = time_arr - t0_plot

            plt.figure(figsize=(5.5, 3.8))
            plt.scatter(
                time_rel,
                flux_arr,
                s=6,                 # smaller points
                alpha=0.5,           # lighter
                color="royalblue",
                edgecolors="black",   # remove black borders (huge visual improvement)
                rasterized=True      # faster rendering for large arrays
            )
            if row.get("y_lim") is not None:
                plt.ylim(row["y_lim"])

            # Mid-transit line (if ephemeris exists)
            if ephem_row is not None and not ephem_row.empty:

                P = ephem_row.iloc[0]["period_days"]
                t0 = ephem_row.iloc[0]["mid_transit_time"]
                
                if pd.isna(P) or pd.isna(t0) or P == 0:
                    P = None
                
                if P is not None:

                    visit_start = time_arr.min()
                    visit_end = time_arr.max()

                    n_start = int(np.floor((visit_start - t0) / P))
                    n_end = int(np.ceil((visit_end - t0) / P))

                    for n in range(n_start, n_end + 1):

                        predicted_mid = t0 + n * P

                        if visit_start <= predicted_mid <= visit_end:
                            plt.axvline(predicted_mid - t0_plot, color="crimson", linewidth=2)

            plt.title(f"Visit {row['visit_id']}")
            plt.xlabel("Time from Visit Start (days)")
            plt.ylabel("Normalized Flux")
            plt.grid(alpha=0.25)
            plt.tight_layout()
            plt.show()

        visit_outputs.append(visit_out)

    visits_container = widgets.VBox(
        visit_outputs,
        layout=widgets.Layout(
            border="1px solid #bbb",
            padding="15px",
            margin="0px 0px 25px 0px"
        )
    )

    return widgets.VBox(
        [title, phase_output, visits_container],
        layout=widgets.Layout(padding="10px")
    )
# ===============================
# 5. CONTROLLER LAYER
# ===============================

def check_enable_button():

    query_valid = any(
        widget.value.strip() != ""
        for widget in INPUT_WIDGETS.values()
    )

    pipe_valid = pipe_chooser.selected is not None or STATE.get("pipe_root")
    cheops_valid = cheops_chooser.selected is not None or STATE.get("drp_root")

    run_button.disabled = not (query_valid and pipe_valid and cheops_valid)


def update_fields(change):

    with dynamic_area:
        clear_output()
        INPUT_WIDGETS.clear()
        run_button.disabled = True

        if change['new'] == "target":
            widget = widgets.Text(description="Target Name:")
            INPUT_WIDGETS['target'] = widget

            widget.observe(lambda x: check_enable_button(), names='value')
            display(widget)

        elif change['new'] == "program":
            widget = widgets.Text(description="Program ID:")
            INPUT_WIDGETS['program'] = widget

            widget.observe(lambda x: check_enable_button(), names='value')
            display(widget)

def update_plot_settings_enabled():

    has_selection = bool(STATE.get("selected_visits"))

    # ---------------------------------
    # Preprocessing section (independent)
    # ---------------------------------

    for prefix, key in [
        ("DRP", "drp_preproc_controls"),
        ("PIPE", "pipe_preproc_controls")
    ]:

        if key not in STATE:
            continue

        controls = STATE[key]

        preprocessing_done = bool(
            STATE.get(f"{prefix}_preprocessing_done")
        )

        # Main checkboxes enabled only if visit selected AND not yet processed
        controls["no_preproc"].disabled = not has_selection or preprocessing_done
        controls["outlier"].disabled = not has_selection or preprocessing_done
        controls["detrend"].disabled = not has_selection or preprocessing_done
        controls["residual"].disabled = not has_selection or preprocessing_done

        # Preprocess button only if selected AND not processed
        controls["preprocess_btn"].disabled = not has_selection or preprocessing_done

        # Reset preprocess only if processed
        controls["reset_preproc_btn"].disabled = not preprocessing_done

        # Parameter fields
        if has_selection and not preprocessing_done:
            controls["nmad_outlier"].disabled = not controls["outlier"].value
            controls["tdens"].disabled = not controls["detrend"].value
            controls["nmad_residual"].disabled = not controls["residual"].value
        else:
            controls["nmad_outlier"].disabled = True
            controls["tdens"].disabled = True
            controls["nmad_residual"].disabled = True

    # ---------------------------------
    # Ephemeris section (independent)
    # ---------------------------------

    for prefix in ["DRP", "PIPE"]:

        preprocessing_done = bool(
            STATE.get(f"{prefix}_preprocessing_done")
        )

        ephem_enabled = has_selection
        
        if f"{prefix}_ephem_query_button" in STATE:
            STATE[f"{prefix}_ephem_query_button"].disabled = not ephem_enabled
        if f"{prefix}_ephem_save_button" in STATE:
            STATE[f"{prefix}_ephem_save_button"].disabled = not ephem_enabled

        if f"{prefix}_ephem_reset_button" in STATE:
            STATE[f"{prefix}_ephem_reset_button"].disabled = True
        

        if f"{prefix}_ephem_row_widgets" in STATE:

            for r in STATE[f"{prefix}_ephem_row_widgets"]:

                r["enable_cb"].disabled = not ephem_enabled

                if not ephem_enabled:
                    r["manual_target"].disabled = True
                    r["period_box"].disabled = True
                    r["mid_box"].disabled = True
        # if f"{prefix}_lc_plot_button" in STATE:
        #     STATE[f"{prefix}_lc_plot_button"].disabled = not has_selection
      

       # ---------------------------------
    # Plot settings dropdown control
    # ---------------------------------

    has_selection = bool(STATE.get("selected_visits"))

    if "drp_aperture_widget" in STATE:
        STATE["drp_aperture_widget"].disabled = not has_selection

    if "pipe_mode_widget" in STATE:
        STATE["pipe_mode_widget"].disabled = not has_selection

                    
                    
def query_exofop(target):

    df = STATE["exofop_catalog_df"]

    if df is None or df.empty:
        return None

    target = target.upper().strip()

    cleaned = (
        target.replace("TIC", "")
              .replace("TOI", "")
              .replace("-", "")
              .strip()
    )

    # ---- Normalize TIC column ----
    tic_clean = (
        df["TIC"]
        .astype(str)
        .str.replace("-", "")
        .str.strip()
    )

    # ---- Normalize TOI column (integer part only) ----
    toi_numeric = pd.to_numeric(df["TOI"], errors="coerce")
    toi_int = toi_numeric.dropna().astype(int).astype(str)

    # Create full-length aligned Series
    toi_clean = pd.Series("", index=df.index)
    toi_clean.loc[toi_numeric.notna()] = toi_numeric.dropna().astype(int).astype(str)

    result = df[
        (tic_clean == cleaned) |
        (toi_clean == cleaned)
    ]

    return result


def query_nasa(target):

    df = STATE.get("nasa_catalog_df")

    if df is None or df.empty:
        return None

    target_clean = (
        target.lower()
              .replace("-", "")
              .replace(" ", "")
              .strip()
    )

    hostname_clean = (
        df["target_name"]
        .astype(str)
        .str.lower()
        .str.replace("-", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )

    result = df[hostname_clean == target_clean]

    return result

def get_catalog_result(target):

    target = target.strip()

    if target.upper().startswith("TIC") or target.upper().startswith("TOI") or target.isdigit():
        return query_exofop(target)
    else:
        return query_nasa(target)
        
    

def execute_query(temp_state):

    drp_root = cheops_chooser.selected or STATE.get("drp_root")
    pipe_root = pipe_chooser.selected or STATE.get("pipe_root")

    if temp_state["mode"] == "target":

        target_name = temp_state["target"]

        loading_view = widgets.VBox([
            widgets.HTML("<h3>Running query... please wait</h3>")
        ])
        
        progress_output = widgets.Output()


        app_container.children = [
            data_section,
            query_section,
            loading_view,
            progress_output
        ]
        
        
        with progress_output:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", FutureWarning)

                df_drp, df_pipe, df_combined = query_table.run_target_query(
                    drp_root=drp_root,
                    pipe_root=pipe_root,
                    target_name=target_name,
                    n_jobs=-1
                )
        
        STATE["df_drp"] = df_drp
        STATE["df_pipe"] = df_pipe
        STATE["df_combined"] = df_combined

      
        results_section = build_results_section(df_drp, df_pipe, df_combined)

        selection_section = build_visit_selection(df_combined)
        
        plot_setting_title = widgets.HTML("<h2>Plot Setting</h2>")

        plot_settings, drp_dropdown, pipe_dropdown = build_plot_settings()
        preproc_section = build_preprocessing_section()
        
        ephem_section = build_ephemerides_section()
        
        lc_section = build_lightcurve_section()

        

        # Store settings widgets in STATE for later use
        STATE["drp_aperture_widget"] = drp_dropdown
        STATE["pipe_mode_widget"] = pipe_dropdown
        

        
        update_plot_settings_enabled()
            

        app_container.children = [
            data_section,
            query_section,
            progress_output,
            results_section,
            selection_section,
            plot_setting_title,
            plot_settings,
            preproc_section,
            ephem_section,
            lc_section


        ]

def restore_main_layout():
    app_container.children = build_base_layout()
        


def run_analysis(b):

    selected_mode = query_mode.value

    temp_state = {"mode": selected_mode}

    if selected_mode in INPUT_WIDGETS:
        temp_state[selected_mode] = INPUT_WIDGETS[selected_mode].value

    def confirm_execution():
        execute_query(temp_state)

    confirmation_section = build_confirmation_section(
        temp_state,
        on_confirm_callback=confirm_execution
    )

    app_container.children = [
        data_section,
        query_section,
        confirmation_section
    ]
    
    
def run_ephemeris_query(b, prefix):

    row_widgets = STATE.get(f"{prefix}_ephem_row_widgets", [])
    output_area = STATE.get(f"{prefix}_ephem_output")

    if not row_widgets:
        return

    with output_area:
        clear_output()

        collected_results = []

        for r in row_widgets:

            target = r["manual_target"].value.strip()

            if not target:
                continue

            result = get_catalog_result(target)

            if result is None or result.empty:
                print(f"No match for {target}")
                continue

            # ---- Always copy immediately ----
            result = result.copy()

            # ---- Force numeric conversion ----
            result["period_days"] = pd.to_numeric(result["period_days"], errors="coerce")
            result["mid_transit_time"] = pd.to_numeric(result["mid_transit_time"], errors="coerce")

            # ---- Drop invalid rows ----
            result = result.dropna(subset=["period_days", "mid_transit_time"])

            # ---- Keep only positive periods ----
            result = result[result["period_days"] > 0]

            if result.empty:
                print(f"No valid ephemeris for {target}")
                continue

            # ---- Add metadata ----
            result["program_type"] = r["program_type"]
            result["program_id"] = r["program_id"]
            result["req_id"] = r["req_id"]
            result["queried_target"] = target

            collected_results.append(result)

            # ---- Select shortest period safely ----
            result = result.sort_values("period_days")
            best_row = result.iloc[0]

            r["period_box"].value = float(best_row["period_days"])
            r["mid_box"].value = float(best_row["mid_transit_time"])

        # =====================================================
        # Final table display
        # =====================================================

        if collected_results:

            final_df = pd.concat(collected_results).reset_index(drop=True)

            ordered_cols = [
                "program_type",
                "program_id",
                "req_id",
                "queried_target",
                "planet_name" if "planet_name" in final_df.columns else None,
                "period_days",
                "mid_transit_time"
            ]

            ordered_cols = [c for c in ordered_cols if c in final_df.columns]

            final_df = final_df[ordered_cols]

            display(final_df)

            STATE[f"{prefix}_ephem_results"] = final_df

            if f"{prefix}_ephem_save_button" in STATE:
                STATE[f"{prefix}_ephem_save_button"].disabled = False

        else:
            print("No ephemeris results found.")
# 6. INITIALIZE APPLICATION
# ===============================

data_section, pipe_chooser, cheops_chooser, default_button, default_output = build_data_source_section()
query_section, query_mode, dynamic_area, run_button = build_query_section()

results_area = widgets.Output()

# Attach controller callbacks
query_mode.observe(update_fields, names='value')
run_button.on_click(run_analysis)

# Default path button
def set_default_paths(b):
    STATE["drp_root"] = DEFAULT_DRP_PATH
    STATE["pipe_root"] = DEFAULT_PIPE_PATH

    with default_output:
        clear_output()
        display(widgets.HTML(
            f"<b>Using Default Paths:</b><br>DRP: {DEFAULT_DRP_PATH}<br>PIPE: {DEFAULT_PIPE_PATH}"
        ))

    check_enable_button()

default_button.on_click(set_default_paths)

# Update enable logic for file choosers
pipe_chooser.register_callback(lambda x: check_enable_button())
cheops_chooser.register_callback(lambda x: check_enable_button())

# Build initial layout
app_container.children = [
    data_section,
    query_section,
    results_area
]

# =========================
# Load ExoFOP
# =========================
exofop_df = pd.read_csv(
    "/media/Ephemerides/exofop_toi_cache.csv",
    usecols=[
        "TIC ID",
        "TOI",
        "Period (days)",
        "Epoch (BJD)"
    ]
)

exofop_df = exofop_df.rename(columns={
    "Period (days)": "period_days",
    "Epoch (BJD)": "mid_transit_time"
})

exofop_df["TIC"] = exofop_df["TIC ID"].astype(str).str.strip()
exofop_df["TOI"] = exofop_df["TOI"].astype(str).str.strip()
# =========================
# Load NASA
# =========================
nasa_df = pd.read_csv(
    "/media/Ephemerides/nasa_pscomppars_cache.csv",
    usecols=[
        "hostname",
        "pl_name",
        "pl_orbper",
        "pl_tranmid"
    ]
)

nasa_df = nasa_df.rename(columns={
    "hostname": "target_name",
    "pl_name": "planet_name",
    "pl_orbper": "period_days",
    "pl_tranmid": "mid_transit_time"
})

# =========================
# Store Clean Catalogs
# =========================
STATE["exofop_catalog_df"] = exofop_df
STATE["nasa_catalog_df"] = nasa_df

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


VBox()